# Entrenamiento del modelo clasificador de riesgo IA

En este notebook entrenamos un modelo baseline para clasificar textos en 4 categorías de riesgo del AI Act:
- Inaceptable
- Alto riesgo
- Riesgo limitado
- Riesgo mínimo

Pasos:
1. Carga de datos (train, validation y test)
2. Vectorización con TF-IDF (bigramas, sublinear_tf)
3. Encoding de features estructuradas (category, context, longitud, num_articles)
4. Combinación de features y entrenamiento (LogisticRegression + class_weight='balanced')
5. Evaluación en validación
6. Guardar artefactos (modelo + vectorizador + encoder)

In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import sys
import os

# Localizar src/classifier/ de forma robusta y ajustar cwd al directorio
# de este notebook para que rutas relativas (datasets/, data/, model/) funcionen
# independientemente de desde donde se lance Jupyter/VS Code.
_cwd = os.getcwd()
_candidates = [
    os.path.join(_cwd, "src", "classifier"),
    os.path.abspath(".."),
    os.path.abspath("."),
]
for _p in _candidates:
    if os.path.isfile(os.path.join(_p, "functions.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        # Cambiar cwd al directorio de este notebook
        os.chdir(os.path.join(_p, "classifier_ultimo_dataset"))
        break

import functions  # noqa: E402
functions.MLFLOW_EXPERIMENT = "clasificador_riesgo_ultimo_dataset"
functions._DATASET_TAGS = {"dataset_type": "ultimo", "dataset_source": "dataset_sintetico_v2"}

## 1. Carga de datos

In [8]:
from pathlib import Path
import pandas as pd

data_dir = Path.cwd() / "data" / "processed"

train_path = data_dir / "train.csv"
val_path   = data_dir / "validation.csv"
test_path  = data_dir / "test.csv"

missing_files = [p.name for p in [train_path, val_path, test_path] if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing processed files: {missing_files}. "
        "Please run the preprocessing pipeline to generate them "
        "(e.g., feature engineering scripts) before training."
    )

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

X_train_text = train_df["text_final"]
X_val_text   = val_df["text_final"]
X_test_text  = test_df["text_final"]

y_train = train_df["etiqueta"]
y_val   = val_df["etiqueta"]
y_test  = test_df["etiqueta"]

# Detectar columnas estructuradas disponibles en el dataset procesado.
# Admite tanto el esquema original (category/context/num_articles)
# como el nuevo (sector/tipo_datos sin num_articles).
_cols = set(train_df.columns) - {"text_final", "etiqueta"}
CAT_COLS = [c for c in ["category", "context", "sector", "tipo_datos"] if c in _cols]
NUM_COLS = [c for c in ["longitud", "num_articles"] if c in _cols]

print(f"Train: {len(train_df)} muestras | Columnas: {list(train_df.columns)}")
print(f"Validation: {len(val_df)} muestras")
print(f"Test: {len(test_df)} muestras")
print(f"CAT_COLS: {CAT_COLS} | NUM_COLS: {NUM_COLS}")

Train: 199 muestras | Columnas: ['text_final', 'longitud', 'sector', 'tipo_datos', 'etiqueta']
Validation: 43 muestras
Test: 43 muestras
CAT_COLS: ['sector', 'tipo_datos'] | NUM_COLS: ['longitud']


## 2. Vectorización TF-IDF

In [9]:
from functions import crear_tfidf

tfidf, X_train_tfidf, X_val_tfidf, X_test_tfidf = crear_tfidf(
    X_train_text, X_val_text, X_test_text,
    max_features=5000,
    ngram_range=(1, 2),
)

Vocabulario TF-IDF: 1700 términos
Forma train: (199, 1700)
Forma validation: (43, 1700)
Forma test: (43, 1700)


## 3. Encoding de features estructuradas

In [10]:
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import csr_matrix, hstack

# Construir X_final combinando TF-IDF + OHE (si hay cols categóricas) + numéricas
parts_train = [X_train_tfidf]
parts_val   = [X_val_tfidf]
parts_test  = [X_test_tfidf]

ohe = None
if CAT_COLS:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    cat_train = ohe.fit_transform(train_df[CAT_COLS])
    cat_val   = ohe.transform(val_df[CAT_COLS])
    cat_test  = ohe.transform(test_df[CAT_COLS])
    parts_train.append(cat_train)
    parts_val.append(cat_val)
    parts_test.append(cat_test)
    print(f"Features categóricas ({CAT_COLS}): {cat_train.shape[1]}")

if NUM_COLS:
    num_train = csr_matrix(train_df[NUM_COLS].values.astype(float))
    num_val   = csr_matrix(val_df[NUM_COLS].values.astype(float))
    num_test  = csr_matrix(test_df[NUM_COLS].values.astype(float))
    parts_train.append(num_train)
    parts_val.append(num_val)
    parts_test.append(num_test)
    print(f"Features numéricas ({NUM_COLS}): {len(NUM_COLS)}")

X_train_final = hstack(parts_train) if len(parts_train) > 1 else parts_train[0]
X_val_final   = hstack(parts_val)   if len(parts_val)   > 1 else parts_val[0]
X_test_final  = hstack(parts_test)  if len(parts_test)  > 1 else parts_test[0]

print(f"Features TF-IDF:    {X_train_tfidf.shape[1]}")
print(f"Total features:     {X_train_final.shape[1]}")

Features categóricas (['sector', 'tipo_datos']): 23
Features numéricas (['longitud']): 1
Features TF-IDF:    1700
Total features:     1724


## 4. Entrenamiento del modelo (LogisticRegression + class_weight='balanced')

In [11]:
from functions import entrenar_modelo_baseline

# class_weight='balanced' compensa el desbalanceo 6.5× (inaceptable:131 vs riesgo_limitado:20)
modelo = entrenar_modelo_baseline(
    X_train_final, y_train,
    X_val_final,   y_val,
    class_weight="balanced",
)

=== Resultados en VALIDACIÓN ===

                 precision    recall  f1-score   support

    alto_riesgo       0.90      0.82      0.86        11
    inaceptable       0.75      0.90      0.82        10
riesgo_limitado       0.83      0.91      0.87        11
  riesgo_minimo       1.00      0.82      0.90        11

       accuracy                           0.86        43
      macro avg       0.87      0.86      0.86        43
   weighted avg       0.87      0.86      0.86        43

F1-score macro (validación): 0.8612


## 5. Guardar artefactos

In [12]:
import joblib
import os

output_dir = "model"
os.makedirs(output_dir, exist_ok=True)

joblib.dump(modelo, os.path.join(output_dir, "modelo_baseline.joblib"))
joblib.dump(tfidf,  os.path.join(output_dir, "tfidf_vectorizer.joblib"))
print("Modelo baseline guardado en:  model/modelo_baseline.joblib")
print("Vectorizador TF-IDF guardado: model/tfidf_vectorizer.joblib")

if ohe is not None:
    joblib.dump(ohe, os.path.join(output_dir, "ohe_encoder.joblib"))
    print("OHE encoder guardado en:      model/ohe_encoder.joblib")

Modelo baseline guardado en:  model/modelo_baseline.joblib
Vectorizador TF-IDF guardado: model/tfidf_vectorizer.joblib
OHE encoder guardado en:      model/ohe_encoder.joblib


In [13]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
from functions import log_mlflow_safe
from sklearn.metrics import f1_score, accuracy_score

y_val_pred = modelo.predict(X_val_final)
extra_features_str = ",".join(CAT_COLS + NUM_COLS)
artifacts_to_log = ["model/modelo_baseline.joblib", "model/tfidf_vectorizer.joblib"]
if ohe is not None:
    artifacts_to_log.append("model/ohe_encoder.joblib")

try:
    log_mlflow_safe(
        run_name="exp0_baseline",
        params={
            "model_type":         "LogisticRegression",
            "model_max_iter":     2000,
            "class_weight":       "balanced",
            "tfidf_max_features": 5000,
            "tfidf_ngram_range":  "(1, 2)",
            "tfidf_sublinear_tf": True,
            "extra_features":     extra_features_str,
            "n_features_total":   X_train_final.shape[1],
        },
        metrics={
            "val_f1_macro": round(f1_score(y_val, y_val_pred, average="macro"), 4),
            "val_accuracy": round(accuracy_score(y_val, y_val_pred), 4),
        },
        artifacts=artifacts_to_log,
        tags={"experimento": "0", "features": f"tfidf+{extra_features_str}"},
    )
    print("✓ Exp 0 (baseline) registrado en MLflow")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

Password obtenida desde variable de entorno local.
MLflow configurado correctamente → https://18.201.64.41/
⚠ MLflow no disponible: API request to https://18.201.64.41/api/2.0/mlflow/experiments/get-by-name failed with timeout exception HTTPSConnectionPool(host='18.201.64.41', port=443): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=clasificador_riesgo_ultimo_dataset (Caused by ConnectTimeoutError(<HTTPSConnection(host='18.201.64.41', port=443) at 0x1ffc10c7200>, 'Connection to 18.201.64.41 timed out. (connect timeout=120)')). To increase the timeout, set the environment variable MLFLOW_HTTP_REQUEST_TIMEOUT (default: 120, type: int) to a larger value.
